# Why the old numbers were wrong

This notebook reproduces the results reported in `notebooks/legacy/` from
runnable code, and puts the corrected numbers next to them. The two defects
below are among the most common mistakes in applied classification, and both
make a model look better rather than worse, which is exactly why they survive
review.

Run it top to bottom in the project environment:

```
pip install -r requirements-dev.txt && pip install -e .
```

In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from stroke.config import SEED
from stroke.data import load_raw, prevalence, split
from stroke.pipeline import build_pipeline

RAW = load_raw()
print(f'{len(RAW)} rows, {int(RAW.stroke.sum())} strokes, prevalence {prevalence(RAW.stroke):.4f}')

5110 rows, 249 strokes, prevalence 0.0487


## The base rate is the whole problem

4.87% of the dataset had a stroke. Any metric that rewards being right *on
average* is dominated by the majority class before a single model is fitted.

In [2]:
base = prevalence(RAW.stroke)
print(f'prevalence                          {base:.4f}')
print(f'accuracy of "nobody has a stroke"   {1 - base:.4f}')

prevalence                          0.0487
accuracy of "nobody has a stroke"   0.9513


## Defect 1 - a 95.7% model that never predicts a stroke

`legacy/log_regression.ipynb` splits with `test_size=0.99`, trains on 1% of
the data, and reports an accuracy of **0.9572**. Its stored confusion matrix
is:

```
[[4652    0]
 [ 208    0]]
```

The entire second column is zero. That model never predicted the positive
class once, and it was reported as 95.7% correct.

That exact result **cannot be reproduced**, because the original set no
`random_state` anywhere - which is defect 4, below. Re-running the same
procedure with a fixed seed lands in the same place for the same reason:

In [3]:
legacy = RAW.dropna().copy()
for col in legacy.select_dtypes(include=['string', 'object']).columns:
    legacy[col] = pd.factorize(legacy[col])[0]

Xl = legacy.drop(columns=['stroke'])
yl = legacy['stroke'].astype(int)

Xtr, Xte, ytr, yte = train_test_split(Xl, yl, test_size=0.99, random_state=SEED)
m = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
pred = m.predict(Xte)

print(f'accuracy           {accuracy_score(yte, pred):.4f}')
print(f'strokes predicted  {int(pred.sum())}')
print(f'strokes present    {int(yte.sum())}')
print()
print(confusion_matrix(yte, pred))

accuracy           0.9521
strokes predicted  40
strokes present    207

[[4620   33]
 [ 200    7]]


Not literally all-negative this time, but functionally the same object: it
identifies a handful of the 207 stroke cases present and scores within half a
percentage point of the original. Whether it catches zero cases or seven is
decided by the unseeded split, and accuracy cannot tell the two apart.

The same predictions, scored with metrics that account for the base rate:

In [4]:
prob = m.predict_proba(Xte)[:, 1]
print(f'accuracy   {accuracy_score(yte, pred):.4f}   <- what was reported')
print(f'PR-AUC     {average_precision_score(yte, prob):.4f}   (no-skill floor {yte.mean():.4f})')
print(f'ROC-AUC    {roc_auc_score(yte, prob):.4f}')
print()
print(classification_report(yte, pred, digits=3, zero_division=0))

accuracy   0.9521   <- what was reported
PR-AUC     0.1086   (no-skill floor 0.0426)
ROC-AUC    0.7105

              precision    recall  f1-score   support

           0      0.959     0.993     0.975      4653
           1      0.175     0.034     0.057       207

    accuracy                          0.952      4860
   macro avg      0.567     0.513     0.516      4860
weighted avg      0.925     0.952     0.936      4860



## Defect 2 - resampling before the split

`legacy/data_utils.py` calls `fit_resample` on the full dataset and splits
afterwards. SMOTE creates synthetic positives by interpolating between
existing positives, so points derived from held-out records land in the
training set and near-duplicates of training points land in the test set.

The two cells below differ in exactly one respect: whether SMOTE runs before
or inside the split. Same estimator, same seed, same data.

In [5]:
num = RAW.select_dtypes(include='number').drop(columns=['stroke'])
cat = pd.get_dummies(RAW.select_dtypes(include='string'), dtype=float)
X = num.join(cat).fillna(num.median(numeric_only=True))
y = RAW['stroke'].astype(int)

# the legacy ordering: resample everything, then split
Xr, yr = SMOTE(random_state=SEED).fit_resample(X, y)
a_tr, a_te, b_tr, b_te = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)
leaky = LogisticRegression(max_iter=5000).fit(a_tr, b_tr)
leaky_prob = leaky.predict_proba(a_te)[:, 1]
leaky_acc = accuracy_score(b_te, leaky.predict(a_te))
leaky_ap = average_precision_score(b_te, leaky_prob)

print('LEGACY ORDERING (resample, then split)')
print(f'  test set: {len(b_te)} rows, {b_te.mean():.1%} positive')
print(f'  accuracy  {leaky_acc:.4f}')
print(f'  PR-AUC    {leaky_ap:.4f}')
print(f'  ROC-AUC   {roc_auc_score(b_te, leaky_prob):.4f}')

LEGACY ORDERING (resample, then split)
  test set: 1945 rows, 49.9% positive
  accuracy  0.7907
  PR-AUC    0.8330
  ROC-AUC   0.8648


In [6]:
# the corrected ordering: split, then resample inside the pipeline
c_tr, c_te, d_tr, d_te = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
honest = ImbPipeline([
    ('resample', SMOTE(random_state=SEED)),
    ('model', LogisticRegression(max_iter=5000)),
]).fit(c_tr, d_tr)
honest_prob = honest.predict_proba(c_te)[:, 1]
honest_acc = accuracy_score(d_te, honest.predict(c_te))
honest_ap = average_precision_score(d_te, honest_prob)

print('CORRECTED ORDERING (split, then resample inside the pipeline)')
print(f'  test set: {len(d_te)} rows, {d_te.mean():.1%} positive')
print(f'  accuracy  {honest_acc:.4f}')
print(f'  PR-AUC    {honest_ap:.4f}')
print(f'  ROC-AUC   {roc_auc_score(d_te, honest_prob):.4f}')
print()
print(f'the leak multiplies PR-AUC by {leaky_ap / honest_ap:.1f}x')

CORRECTED ORDERING (split, then resample inside the pipeline)
  test set: 1022 rows, 4.9% positive
  accuracy  0.7632
  PR-AUC    0.1830
  ROC-AUC   0.8138

the leak multiplies PR-AUC by 4.6x


The leaked test set is roughly balanced; the honest one keeps the population
rate. That gap is the entire difference between the two cells - the model,
the seed and the data are identical.

It is also why the legacy notebooks show a test set of 958 negatives to 922
positives. A balanced test set drawn from a 4.87%-positive population is the
visible symptom of the bug.

## Defect 3 - `dropna()` is not neutral here

Both legacy cleaning modules drop the 201 rows with a missing BMI. Those rows
are not a random sample of the dataset.

In [7]:
miss = RAW[RAW.bmi.isna()]
pres = RAW[RAW.bmi.notna()]

table = pd.DataFrame(
    {
        'rows': [len(pres), len(miss), len(RAW)],
        'strokes': [
            int(pres.stroke.sum()),
            int(miss.stroke.sum()),
            int(RAW.stroke.sum()),
        ],
        'stroke_rate': [pres.stroke.mean(), miss.stroke.mean(), RAW.stroke.mean()],
    },
    index=['bmi present', 'bmi missing', 'total'],
)
print(table.round(4))

lost = int(RAW.stroke.sum() - pres.stroke.sum())
print()
print(f'dropna() discards {lost} of {int(RAW.stroke.sum())} stroke cases ({lost / RAW.stroke.sum():.1%} of all positives)')
print(f'ratio of stroke rates: {miss.stroke.mean() / pres.stroke.mean():.1f}x')

             rows  strokes  stroke_rate
bmi present  4909      209       0.0426
bmi missing   201       40       0.1990
total        5110      249       0.0487

dropna() discards 40 of 249 stroke cases (16.1% of all positives)
ratio of stroke rates: 4.7x


Missing BMI carries roughly 4.7x the stroke rate. Whatever produced that gap
is correlated with the outcome, so the missingness itself is a feature. The
current pipeline imputes BMI on the training fold and adds an indicator column
rather than discarding those patients.

## Defect 4 - none of it was reproducible

`legacy/data_utils.py` passes no `random_state` to `train_test_split`, to
`SMOTE`, to `ADASYN`, or to any estimator. Every reported figure therefore
changed on each re-run, and the numbers stored in the notebooks could not be
regenerated by the code that produced them.

The spread below is what that costs. It is the same pipeline, evaluated ten
times, differing only in the split seed:

In [8]:
scores = []
for s in range(10):
    xa, xb, ya, yb = split(RAW, seed=s)
    fitted = build_pipeline('logistic_regression', 'none', s).fit(xa, ya)
    scores.append(average_precision_score(yb, fitted.predict_proba(xb)[:, 1]))

scores = pd.Series(scores)
print(f'PR-AUC over 10 seeds: min {scores.min():.4f}  max {scores.max():.4f}  spread {scores.max() - scores.min():.4f}')
print(f'mean {scores.mean():.4f} +/- {scores.std():.4f}')
print()
print('A single unseeded run could have reported any value in that range.')

PR-AUC over 10 seeds: min 0.1797  max 0.3029  spread 0.1232
mean 0.2319 +/- 0.0380

A single unseeded run could have reported any value in that range.


## What the corrected pipeline reports

Same data, leak-free, with metrics that survive the imbalance.

In [9]:
X_train, X_test, y_train, y_test = split(RAW, seed=SEED)
pipe = build_pipeline('logistic_regression', 'none', SEED).fit(X_train, y_train)
p = pipe.predict_proba(X_test)[:, 1]
corrected_acc = accuracy_score(y_test, (p >= 0.5).astype(int))

print(f'test set: {len(y_test)} rows, {y_test.mean():.2%} positive ({int(y_test.sum())} cases)')
print()
print(f'PR-AUC   {average_precision_score(y_test, p):.4f}   (no-skill floor {y_test.mean():.4f})')
print(f'ROC-AUC  {roc_auc_score(y_test, p):.4f}')
print(f'accuracy {corrected_acc:.4f}   <- still ~95%, and still meaningless')

test set: 1022 rows, 4.89% positive (50 cases)

PR-AUC   0.2575   (no-skill floor 0.0489)
ROC-AUC  0.8425
accuracy 0.9521   <- still ~95%, and still meaningless


Accuracy is essentially *unchanged* for the corrected model, because at a 0.5
threshold it also predicts almost nobody as positive. That is the point:
accuracy did not distinguish the broken model from the working one, and never
could have.

## Summary

In [10]:
summary = pd.DataFrame(
    [
        ['legacy: all-negative model', accuracy_score(yte, pred),
         average_precision_score(yte, prob), 'never predicts a stroke'],
        ['legacy: SMOTE before split', leaky_acc, leaky_ap,
         'test set leaked and rebalanced'],
        ['corrected: SMOTE in pipeline', honest_acc, honest_ap, 'honest'],
        ['corrected: no resampling', corrected_acc,
         average_precision_score(y_test, p), 'honest, and the reported model'],
    ],
    columns=['pipeline', 'accuracy', 'pr_auc', 'note'],
)
summary.round(4)

,pipeline,accuracy,pr_auc,note
0,legacy: all-negative model,0.9521,0.1086,never predicts a stroke
1,legacy: SMOTE before split,0.7907,0.8330,test set leaked and rebalanced
2,corrected: SMOTE in pipeline,0.7632,0.1830,honest
3,corrected: no resampling,0.9521,0.2575,"honest, and the reported model"


Read the `accuracy` column on its own and every row looks like a success.
Read `pr_auc` against the no-skill floor of 0.049 and the ordering is
completely different.

Full results, confidence intervals and the operating-point analysis are in the
project README, produced by `python -m stroke.train`.